# Create APVV Awards

Creates OpenAlex award rows from the Slovak Research and Development Agency (APVV) official funded-project database.

**Data source:** `https://www.apvv.sk/databaza-financovanych-projektov.html`, APVV's own public "Databáza financovaných projektov" table. The local exporter walks the paginated table and preserves each project's real APVV project number, project title, recipient organization, approved EUR support, call label, scientific field, and final-report PDF link when present.
**S3 location:** `s3a://openalex-ingest/awards/apvv/apvv_projects.parquet`
**OpenAlex funder:** `F4320323251` - Agentúra na Podporu Výskumu a Vývoja (no ROR/DOI in OpenAlex), country `SK`.
**Provenance:** `apvv_funded_projects`.
**Priority:** 220.

**Schema choices:**
- One row per unique APVV project number; local full validation found 5,431 rows and 5,431 unique IDs.
- Stable `funder_award_id = project_number` (for example `APVV-15-0000` / `APVT-10-035702`), preserving the real citable APVV grant identifier for WorkAwards matching.
- `amount` is parsed from the first EUR value in `Schválená podpora v €`; historical Slovak-koruna parenthetical values stay in raw `amount_raw` only. `currency = EUR` when amount is positive.
- APVV publishes recipient organizations, not PIs. `lead_investigator` is therefore org-level: given/family/orcid NULL and `affiliation.name = recipient_organization`. `affiliation.country` remains NULL because the table does not publish per-recipient country.
- APVV publishes call labels with years but no day-level start/end dates. `start_year` is populated from the call label; `start_date`, `end_date`, and `end_year` remain NULL.
- `description` remains NULL because the table does not publish abstracts/descriptions, and scientific field/call labels are not substituted as prose descriptions.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.apvv_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/apvv/apvv_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS total_apvv_projects FROM openalex.awards.apvv_raw;


## Step 1.5: Inspect Raw Data, Money Fields, and Key Uniqueness

Expected source columns: `funder_award_id`, `project_number`, `display_name`, `description`, `recipient_organization`, `amount`, `currency`, `amount_raw`, `funder_scheme`, `scientific_field`, `funding_type`, `start_year`, `end_year`, `start_date`, `end_date`, `final_report_text`, `final_report_url`, `landing_page_url`, `source_page_url`, `provenance`, `funder_id`, `funder_display_name`, `downloaded_at`.


In [ ]:
%sql
DESCRIBE openalex.awards.apvv_raw;


In [ ]:
%sql
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'apvv_raw'
  AND LOWER(column_name) RLIKE 'amount|currency|year|scheme|field|report'
ORDER BY column_name;


In [ ]:
%sql
SELECT funder_award_id, display_name, recipient_organization, amount, currency,
       funder_scheme, scientific_field, start_year, landing_page_url
FROM openalex.awards.apvv_raw
LIMIT 10;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(recipient_organization) AS has_recipient,
    COUNT(funder_scheme) AS has_scheme,
    COUNT(scientific_field) AS has_scientific_field,
    COUNT(TRY_CAST(amount AS DOUBLE)) AS has_amount,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS eur_total,
    COUNT(TRY_CAST(start_year AS INT)) AS has_start_year,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    COUNT(final_report_url) AS has_final_report_url,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    ROUND(try_divide(COUNT(TRY_CAST(amount AS DOUBLE)), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.apvv_raw;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) AS awards, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS eur_total
FROM openalex.awards.apvv_raw
GROUP BY funder_scheme
ORDER BY awards DESC
LIMIT 25;


In [ ]:
%sql
SELECT scientific_field, COUNT(*) AS awards, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS eur_total
FROM openalex.awards.apvv_raw
GROUP BY scientific_field
ORDER BY awards DESC;


In [ ]:
%sql
SELECT funding_type, COUNT(*) AS awards, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS eur_total
FROM openalex.awards.apvv_raw
GROUP BY funding_type
ORDER BY awards DESC;


In [ ]:
%sql
SELECT start_year, COUNT(*) AS awards, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS eur_total
FROM openalex.awards.apvv_raw
GROUP BY start_year
ORDER BY TRY_CAST(start_year AS INT);


In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS n
FROM openalex.awards.apvv_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY n DESC, funder_award_id
LIMIT 20;


## Step 1.6: Fail-Fast - Verify APVV Funder Exists

F4320323251 is a Crossref-registered `F4320*` funder. This assertion prevents a silent zero-row transform if the funder dimension is unexpectedly missing.


In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'Expected exactly one APVV funder row for F4320323251') AS apvv_funder_exists
FROM openalex.common.funder
WHERE funder_id = 4320323251;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.apvv_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT
        4320323251 AS funder_id,
        COALESCE(MAX(display_name), 'Agentúra na Podporu Výskumu a Vývoja') AS display_name,
        MAX(ror_id) AS ror_id,
        MAX(doi) AS doi
    FROM openalex.common.funder
    WHERE funder_id = 4320323251
),
raw_prepped AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS amount_double,
        CASE WHEN TRY_CAST(start_year AS INT) BETWEEN 1800 AND 2100 THEN TRY_CAST(start_year AS INT) END AS start_year_int,
        NULLIF(TRIM(display_name), '') AS display_name_norm,
        NULLIF(TRIM(description), '') AS description_norm,
        NULLIF(TRIM(recipient_organization), '') AS recipient_organization_norm,
        NULLIF(TRIM(funder_scheme), '') AS funder_scheme_norm,
        NULLIF(TRIM(funding_type), '') AS funding_type_norm,
        NULLIF(TRIM(landing_page_url), '') AS landing_page_url_norm
    FROM openalex.awards.apvv_raw
)
SELECT
    abs(xxhash64(CONCAT('4320323251:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS id,
    rp.display_name_norm AS display_name,
    rp.description_norm AS description,
    4320323251 AS funder_id,
    rp.funder_award_id,
    CASE WHEN rp.amount_double > 0 THEN rp.amount_double ELSE NULL END AS amount,
    CASE WHEN rp.amount_double > 0 THEN 'EUR' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    COALESCE(rp.funding_type_norm, 'research') AS funding_type,
    rp.funder_scheme_norm AS funder_scheme,
    'apvv_funded_projects' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    CASE
        WHEN rp.start_year_int > YEAR(current_date()) + 1 THEN NULL
        ELSE rp.start_year_int
    END AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN rp.recipient_organization_norm IS NOT NULL THEN
            struct(
                CAST(NULL AS STRING) AS given_name,
                CAST(NULL AS STRING) AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    rp.recipient_organization_norm AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    rp.landing_page_url_norm AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320323251:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_prepped rp
CROSS JOIN funder_resolved f
WHERE rp.funder_award_id IS NOT NULL
  AND rp.display_name_norm IS NOT NULL;


In [ ]:
%sql
SELECT COUNT(*) AS transformed_awards FROM openalex.awards.apvv_awards;


In [ ]:
%sql
SELECT id, COUNT(*) AS n
FROM openalex.awards.apvv_awards
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY n DESC
LIMIT 20;


In [ ]:
%sql
SELECT funder_award_id, display_name, amount, currency, funder_scheme,
       start_year, lead_investigator.affiliation.name AS recipient, landing_page_url
FROM openalex.awards.apvv_awards
LIMIT 10;


## Step 3: Delete Existing APVV Rows, Then Insert

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'apvv_funded_projects' AND priority = 220;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    220 AS priority
FROM openalex.awards.apvv_awards;


## Step 6: Verification

In [ ]:
%sql
SELECT COUNT(*) AS inserted_rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'apvv_funded_projects' AND priority = 220;


In [ ]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(SUM(amount), 0) AS eur_total,
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'apvv_funded_projects' AND priority = 220;


In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS eur_total
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'apvv_funded_projects' AND priority = 220
GROUP BY funding_type, funder_scheme
ORDER BY awards DESC
LIMIT 25;


In [ ]:
%sql
SELECT funder_award_id, display_name, amount, currency, funder_scheme,
       start_year, lead_investigator.affiliation.name AS recipient, landing_page_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'apvv_funded_projects' AND priority = 220
ORDER BY funder_award_id
LIMIT 25;


In [ ]:
%sql
SELECT id, funder_award_id, works_api_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'apvv_funded_projects' AND priority = 220
LIMIT 10;


## Handoff / Admin Notes

Contractor-complete only. The contractor has no S3 or Databricks access.

Admin follow-up:
- Upload `/tmp/apvv_projects.parquet` or rerun `scripts/local/apvv_to_s3.py --allow-shrink` with AWS credentials to write `s3://openalex-ingest/awards/apvv/apvv_projects.parquet`.
- Run this notebook on Databricks and inspect the Step 6 verification cells.
- After QA, update tracker status beyond Step 4 and only then consider scheduled job wiring.
